### *[MIST 4600] Fall 2026 · Sample project*
# **Family Game Night Picker**

**My app helps a parent decide which game to buy for family game night by checking each game's age rating, whether it runs on the family's Mac, and whether the family can play it together.**

This is a sample GitHub project for MIST 4600. It shows what a finished project can look like: how the cells are organized, how text cells explain each step, and how the answers add up to a recommendation. Your project answers **your own** question with **your own** code.

| # | Sub-question | Tools |
|---|---|---|
| 1 | **Age:** how many well-known games are rated for all ages, and how do their reviews compare with age-rated games? | pandas |
| 2 | **Mac:** of the well-reviewed all-ages games, how many run on a Mac? | pandas |
| 3 | **Together:** can a family play each shortlisted game together on one computer, only online, or only alone? | Gemini LLM API |

**Requirements checklist:** a user and a decision (above) · 3 sub-questions · a function for each sub-question (`compare_age_groups`, `count_mac_games`, `make_shortlist`) · a class the app uses (`GameNightPick`, Cell 5) · an outside tool (Gemini) · runs from top to bottom · a recommendation at the end.

---
## Before you run it

1. Put this notebook and `steam_games.csv` in the same folder in your Google Drive.
2. Add your Gemini key to Colab Secrets (key icon on the left) with the name `GOOGLE_API_KEY`, and turn on **Notebook access**. Get the key at [aistudio.google.com](https://aistudio.google.com) with a personal Gmail account.
3. Runtime → **Run all**. Cell 6 calls Gemini 5 times with a 4-second pause between calls, so it takes about 30 seconds.

## Setup: find the data in Drive

Connects to Google Drive and finds the folder that holds `steam_games.csv`. Cell 1 uses that folder, `FOLDER`, to read the file.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Put it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    print("✅ Found the data in:", FOLDER)

## Cell 1 · Setup: libraries, data, and Gemini

Imports the libraries, reads the 1,000 Steam games into `games`, and connects to Gemini. The key is read from Colab Secrets, so it never appears in the code (this repository is public).

In [ ]:
import pandas as pd
import time
from google import genai
from google.colab import userdata

games = pd.read_csv(FOLDER + "steam_games.csv")
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"

print(games.shape)

## Cell 2 · Sub-question 1: Age

`age_group` turns a game's minimum age (`required_age`) into a group: **all ages** when it is 0, **age-rated** otherwise.

`compare_age_groups` answers the sub-question: it labels every game with `.apply`, keeps only well-known games (at least 100,000 reviews), counts each group, and compares their average percent of positive reviews. Do family games have to give up on quality? It returns the well-known games, `known`, for the next step.

In [ ]:
def age_group(age):
    if age == 0:
        return "all ages"
    else:
        return "age-rated"

def compare_age_groups(games):
    games["age_group"] = games["required_age"].apply(age_group)
    known = games[games["num_reviews_total"] >= 100000]
    print(known["age_group"].value_counts())
    print(known.groupby("age_group")["pct_pos_total"].mean().round(1))
    return known

known = compare_age_groups(games)

## Cell 3 · Sub-question 2: Mac

The family plays on a Mac. `count_mac_games` keeps the well-reviewed all-ages games (90% positive or more) and counts how many run on a Mac. The last line counts Mac support for all well-known games, to compare. It returns the well-reviewed all-ages games, `family`.

In [ ]:
def count_mac_games(known):
    family = known[known["age_group"] == "all ages"]
    family = family[family["pct_pos_total"] >= 90]
    print(len(family))
    print(family["mac"].value_counts())
    print(known["mac"].value_counts())
    return family

family = count_mac_games(known)

## Cell 4 · Sub-question 3: Play together (Gemini)

`get_play_style` sends a game's store description to Gemini and asks how a family can play it: **couch** (together on one computer), **online** (only with other people online), or **solo** (only alone).

* `time.sleep(4)` waits 4 seconds before each call, to stay under the free tier's limit on requests per minute.
* The answer is cleaned with `.strip().lower().strip(".")`, so `"Couch.\n"` becomes `"couch"`.
* Only the three answers in `PLAY` are accepted; anything else becomes `"unknown"`, so one odd answer can't break the results.

In [ ]:
PLAY = ["couch", "online", "solo"]

def get_play_style(description):
    time.sleep(4)
    prompt = f"Can a family play this game together on one computer (couch), only online with other people (online), or only alone (solo)? Answer with one word: couch, online, or solo. Description: {description}"
    response = client.models.generate_content(
        model=MODEL, contents=prompt)
    answer = response.text.strip().lower().strip(".")
    if answer in PLAY:
        return answer
    return "unknown"

Try it on one game:

In [ ]:
print(games["name"][0])
print(get_play_style(games["short_description"][0]))

## Cell 5 · A class for each pick

Each game on the shortlist becomes a `GameNightPick` object with its name, price, and play style.

* `good_for_game_night()` returns `True` if the family can play it together on one computer.
* `card()` returns one line of text for the parent, showing "free" for free games.

In [ ]:
class GameNightPick:
    def __init__(self, name, price, play):
        self.name = name
        self.price = price
        self.play = play

    def good_for_game_night(self):
        return self.play == "couch"

    def card(self):
        if self.price == 0:
            cost = "free"
        else:
            cost = f"${self.price:.2f}"
        return f"{self.name}: {cost}, plays {self.play}"

Try the class with a made-up pick:

In [ ]:
test = GameNightPick("Test Game", 0, "couch")
print(test.card())
print(test.good_for_game_night())

## Cell 6 · Sub-question 3 and the shortlist

1. `make_shortlist` keeps the well-reviewed all-ages games that run on a Mac, takes the 5 with the most reviews, and labels how each is played with `get_play_style` (5 Gemini calls, not 1,000, to stay inside the free tier).
2. Each game becomes a `GameNightPick`; we print its card and list the ones the family can play together.
3. The shortlist is saved to `game_night.csv`.

In [ ]:
def make_shortlist(family):
    mac_family = family[family["mac"] == True]
    shortlist = mac_family.sort_values("num_reviews_total", ascending=False).head(5).copy()
    shortlist["play"] = shortlist["short_description"].apply(get_play_style)
    return shortlist

shortlist = make_shortlist(family)

picks = []
# AI helped: to_dict("records") turns the table into a list of
# dictionaries, one per row (like the Week 7 list of dictionaries)
for game in shortlist.to_dict("records"):
    picks.append(GameNightPick(game["name"], game["price"], game["play"]))

for pick in picks:
    print(pick.card())

print("Best for game night:")
for pick in picks:
    if pick.good_for_game_night():
        print("-", pick.name)
shortlist.to_csv("game_night.csv", index=False)

---
## The recommendation

**Sub-question 1 (Cell 2):** most well-known games (119 of 169) are rated for all ages, and their reviews are about as good as age-rated games' (88% vs. 86% positive). A family doesn't have to give up on quality.

**Sub-question 2 (Cell 3):** 40 of the 67 well-reviewed all-ages games run on a Mac (60%), compared with 65 of the 169 well-known games overall (38%). Family-friendly games are more likely to run on a Mac.

**Sub-question 3 (Cell 6):** the shortlist is Terraria, Garry's Mod, Stardew Valley, Euro Truck Simulator 2, and Unturned. Gemini labels how each is played; the "Best for game night" list shows the ones the family can play together on one computer.

**For the parent:** buy one of the games on the "Best for game night" list. If none comes up as couch, pick the cheapest well-reviewed game and take turns.

*(Your project should end the same way: answer your user's decision in a few sentences, using your own numbers.)*